# Lab 2: 语音识别 — Qwen3-ASR + OpenVINO

Qwen3-ASR 是通义千问团队推出的自动语音识别模型，支持 52 种语言和方言的语音识别和语言识别。该模型基于 Qwen3-Omni 的强大音频理解能力和大规模语音训练数据，在开源 ASR 模型中达到了最先进的性能。

**主要特点:**
- 🌍 支持 30 种语言和 22 种中文方言的语音识别
- ⚡ 0.6B 版本在保持高精度的同时实现极高吞吐量
- 🎤 支持流式/离线统一推理
- 📏 支持长音频转写

在本实验中，我们将：
1. 从 ModelScope 下载预转换的 OpenVINO 模型
2. 使用 OpenVINO 加载并运行语音识别
3. 体验交互式 Gradio 语音识别演示

#### 目录：
- [下载模型](#下载模型)
- [选择推理设备](#选择推理设备)
- [加载模型](#加载模型)
- [运行语音识别](#运行语音识别)
- [交互式演示](#交互式演示)

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载已转换的 Qwen3-ASR-0.6B OpenVINO 模型。如果模型已存在则跳过下载。

In [1]:
from pathlib import Path

model_dir = Path("Qwen3-ASR-0.6B-fp16-ov")

if not model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-ASR-0.6B-fp16-ov", local_dir=str(model_dir))
    print(f"模型已下载到: {model_dir}")
else:
    print(f"模型已存在: {model_dir}，跳过下载")

模型已存在: Qwen3-ASR-0.6B-fp16-ov，跳过下载


## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。CPU 兼容性最好，GPU 可在支持的硬件上提供加速。

In [5]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

## 加载模型
[返回目录 ⬆️](#目录：)

使用 `OVQwen3ASRModel` 加载 OpenVINO 模型。该 wrapper 提供与原始 `Qwen3ASRModel` 相同的 API 接口。

In [6]:
from qwen_3_asr_helper import OVQwen3ASRModel

ov_model = OVQwen3ASRModel.from_pretrained(
    model_dir=str(model_dir),
    device=device.value,
    max_inference_batch_size=32,
    max_new_tokens=256,
)

print(f"支持的语言: {ov_model.get_supported_languages()}")

Loading OV models from Qwen3-ASR-0.6B-fp16-ov on CPU…
All models loaded ✅
Processor loaded ✅
OVQwen3ASRModel ready ✅
支持的语言: ['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']


## 运行语音识别
[返回目录 ⬆️](#目录：)

让我们用一段英文音频来测试模型。模型支持多种音频格式（wav, mp3, flac），也可以接受文件路径、URL 或 NumPy 数组。

In [7]:
import urllib.request

# 下载测试音频
sample_audio_en = Path("sample_en.wav")

if not sample_audio_en.exists():
    print("正在下载英文示例音频...")
    urllib.request.urlretrieve("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio_en)
    print(f"已下载: {sample_audio_en}")

# 运行转写
results = ov_model.transcribe(
    audio=str(sample_audio_en),
    language=None,  # 自动检测语言
)

print(f"\n检测到的语言: {results[0].language}")
print(f"转写结果: {results[0].text}")


检测到的语言: English
转写结果: Mhm. Oh yeah, yeah. He wasn't even that big when I started listening to him, but and his solo music didn't do overly well, but he did very well when he started writing for other people.


## 交互式演示
[返回目录 ⬆️](#目录：)

启动 Gradio 交互式演示，你可以：
- 上传音频文件（WAV, MP3, FLAC 等）
- 使用麦克风录音
- 选择目标语言或使用自动检测
- 查看转写结果和推理指标

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, example_dir=".")

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)

D:\ai-contest\modelscope-workshop\lab2-speech-recognition\gradio_helper.py:296: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title="Qwen3-ASR with OpenVINO") as demo:
HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.
